In [1]:
import pandas as pd
import numpy as np
import ast

from nltk.stem.porter import PorterStemmer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

In [2]:
# =========================
# LOAD DATA
# =========================

movies = pd.read_csv('tmdb_5000_movies.csv')
credits = pd.read_csv('tmdb_5000_credits.csv')

In [3]:

# =========================
# MERGE
# =========================

movies = movies.merge(credits, on='title')


In [4]:

movies = movies[[
    'movie_id',
    'title',
    'overview',
    'genres',
    'keywords',
    'cast',
    'crew'
]]

In [5]:
# =========================
# REMOVE NULL VALUES
# =========================

movies.dropna(inplace=True)

In [6]:
# =========================
# FUNCTIONS
# =========================

def convert(obj):
    L = []
    for i in ast.literal_eval(obj):
        L.append(i['name'])
    return L


def convert_cast(obj):
    L = []
    for i in ast.literal_eval(obj)[:5]:
        L.append(i['name'].replace(" ", ""))
    return L


def fetch_director(obj):
    L = []
    for i in ast.literal_eval(obj):
        if i['job'] == 'Director':
            L.append(i['name'].replace(" ", ""))
    return L

In [7]:
# =========================
# APPLY FUNCTIONS
# =========================

movies['genres'] = movies['genres'].apply(convert)

movies['keywords'] = movies['keywords'].apply(convert)

movies['cast'] = movies['cast'].apply(convert_cast)

movies['crew'] = movies['crew'].apply(fetch_director)

movies['overview'] = movies['overview'].apply(lambda x: x.split())

In [8]:
# =========================
# CREATE TAGS
# =========================

movies['tags'] = (
    movies['genres'] * 3 +
    movies['keywords'] * 2 +
    movies['cast'] * 2 +
    movies['crew'] * 3 +
    movies['overview']
)

In [9]:

# =========================
# NEW DATAFRAME
# =========================

new_df = movies[['movie_id', 'title', 'tags']].copy()


In [10]:

# =========================
# CONVERT LIST TO STRING
# =========================

new_df['tags'] = new_df['tags'].apply(lambda x: " ".join(x))


In [11]:

# =========================
# LOWERCASE
# =========================

new_df['tags'] = new_df['tags'].apply(lambda x: x.lower())

In [12]:
# =========================
# STEMMING
# =========================

ps = PorterStemmer()


def stem(text):
    return " ".join([ps.stem(i) for i in text.split()])


new_df['tags'] = new_df['tags'].apply(stem)


In [13]:
new_df['tags'][0]

'action adventur fantasi scienc fiction action adventur fantasi scienc fiction action adventur fantasi scienc fiction cultur clash futur space war space coloni societi space travel futurist romanc space alien tribe alien planet cgi marin soldier battl love affair anti war power relat mind and soul 3d cultur clash futur space war space coloni societi space travel futurist romanc space alien tribe alien planet cgi marin soldier battl love affair anti war power relat mind and soul 3d samworthington zoesaldana sigourneyweav stephenlang michellerodriguez samworthington zoesaldana sigourneyweav stephenlang michellerodriguez jamescameron jamescameron jamescameron in the 22nd century, a parapleg marin is dispatch to the moon pandora on a uniqu mission, but becom torn between follow order and protect an alien civilization.'

In [16]:
# =========================
# VECTORIZATION
# =========================

tfidf = TfidfVectorizer(max_features=5000, stop_words='english')

vectors = tfidf.fit_transform(new_df['tags']).toarray()


In [17]:
# =========================
# COSINE SIMILARITY
# =========================

similarity = cosine_similarity(vectors)


In [18]:

# =========================
# RECOMMEND FUNCTION
# =========================

def recommend(movie):

    movie_index = new_df[new_df['title'] == movie].index[0]

    distances = similarity[movie_index]

    movie_list = sorted(
        list(enumerate(distances)),
        reverse=True,
        key=lambda x: x[1]
    )[1:6]

    print(f"\nRecommendations for {movie}:\n")

    for i in movie_list:
        print(new_df.iloc[i[0]].title)



In [21]:

recommend('Frozen')


Recommendations for Frozen:

Aladdin
The Snow Queen
Snow White and the Seven Dwarfs
Pokémon: Spell of the Unknown
Return to Never Land


In [22]:
import pickle

In [32]:
pickle.dump(new_df,open('movies.pkl','wb'))

In [25]:
new_df['title'].values

array(['Avatar', "Pirates of the Caribbean: At World's End", 'Spectre',
       ..., 'Signed, Sealed, Delivered', 'Shanghai Calling',
       'My Date with Drew'], shape=(4806,), dtype=object)

In [31]:
pickle.dump(similarity, open('similarity.pkl', 'wb'))